In [1]:
import os
import sys
import argparse

import numpy as np
import matplotlib.pyplot as plt

import torch

import dolfinx
import dolfinx.fem.petsc
import ufl
from mpi4py import MPI
import basix.ufl

repo_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(repo_path)

from utils import load_yaml, load_pkl, load_npy, format_elapsed_time, timing, plot_complex_valued_function, plot_real_valued_function, evaluate_expression

from scifem import create_real_functionspace

import torch
import torch.nn as nn
import torch.nn.init as init
import numpy as np
from petsc4py import PETSc
import scifem


from utils import project, norm_L2, convert_petsc_mat_to_torch_sparse_coo_tensor, convert_weight_to_tensor
from typing import Optional
import pickle
import time
from tqdm import tqdm
import matplotlib.ticker as ticker

from data_generation.differential_equations import PoissonSetup1LeastSquares
from data_generation.probability_measure import generate_random_p

----------------------------------------
2025-11-25 22:17:15 - Start Program
----------------------------------------


In [2]:
mesh_args = {
    'lower_left_x': 0.0,  # x coordinate of the lower left corner of the domain
    'lower_left_y': 0.0, # y coordinate of the lower left corner of the domain
    'upper_right_x': 1.0,  # x coordinate of the upper right corner of the domain
    'upper_right_y': 1.0,  # y coordinate of the upper right corner of the domain
    'num_x': 64,   # Number of cells in the x-axis
    'num_y': 64,  # Number of cells in the y-axis
    'mesh_cell_type': 'triangle'  # Type of mesh cell (triangle or quadrilateral)
}

In [22]:
function_space_args = {
    "p": {
        "family": "DG",
        "degree": 0
    },
    "u": {
        "family": "CG",
        "degree": 2
    },
    "sigma": {
        "family": "RT",
        "degree": 2
    },
    "w": {
        "family": "CG",
        "degree": 2
    },
    "q": {
        "family": "CG",
        "degree": 2
    }
}

In [4]:
function_space_finer_args = {
    "p": {
        "family": "DG",
        "degree": 0
    },
    "u": {
        "family": "CG",
        "degree": 4
    },
    "sigma": {
        "family": "RT",
        "degree": 4
    },
    "w": {
        "family": "CG",
        "degree": 4
    },
    "q": {
        "family": "CG",
        "degree": 4
    }
}


In [5]:
print(f'mesh args: {mesh_args}')
print(f'function space args: {function_space_args}')
print(f'function space finer args: {function_space_finer_args}')

mesh args: {'lower_left_x': 0.0, 'lower_left_y': 0.0, 'upper_right_x': 1.0, 'upper_right_y': 1.0, 'num_x': 64, 'num_y': 64, 'mesh_cell_type': 'triangle'}
function space args: {'p': {'family': 'DG', 'degree': 0}, 'u': {'family': 'CG', 'degree': 1}, 'sigma': {'family': 'RT', 'degree': 1}, 'w': {'family': 'CG', 'degree': 1}, 'q': {'family': 'CG', 'degree': 1}}
function space finer args: {'p': {'family': 'DG', 'degree': 0}, 'u': {'family': 'CG', 'degree': 4}, 'sigma': {'family': 'RT', 'degree': 4}, 'w': {'family': 'CG', 'degree': 4}, 'q': {'family': 'CG', 'degree': 4}}


In [23]:
poisson_least_squares = PoissonSetup1LeastSquares(mesh_args, function_space_args)
poisson_least_squares_finer = PoissonSetup1LeastSquares(mesh_args, function_space_finer_args)

In [24]:
mesh = poisson_least_squares.mesh
Vh = poisson_least_squares.Vh

In [32]:
# Make sure two function spaces are defined on the same mesh
u_element = basix.ufl.element(family=function_space_finer_args["u"]["family"],
                              cell=mesh_args["mesh_cell_type"],
                              degree=function_space_finer_args["u"]["degree"])
sigma_element = basix.ufl.element(family=function_space_finer_args["sigma"]["family"],
                                  cell=mesh_args["mesh_cell_type"],
                                  degree=function_space_finer_args["sigma"]["degree"])
sigma_u_element = basix.ufl.mixed_element([sigma_element, u_element])
finer_Vh = {
    'sigma_u': dolfinx.fem.functionspace(mesh, sigma_u_element)
}

In [9]:
num_samples = 100
seed = 0

In [10]:
# Generate random samples
batch_p, batch_mu = generate_random_p(Vh['p'], num_samples=num_samples, seed=seed)

p_dim = len(dolfinx.fem.Function(Vh['p']).x.array)
p_dof = np.zeros((num_samples, p_dim))
for i in range(num_samples):
    p_dof[i, :] = batch_p[i].x.array

unique_data, indices, counts = np.unique(p_dof, axis=0, return_index=True, return_counts=True)
# Check duplicates
has_duplicates = np.any(counts > 1)
print("Has duplicates:", has_duplicates)

100%|██████████| 100/100 [00:05<00:00, 17.83it/s]


Has duplicates: False


In [25]:
# Solve auxiliary problems
poisson_least_squares.q = poisson_least_squares.solve_q()
poisson_least_squares.w = poisson_least_squares.solve_w()

In [26]:
dtype = 'float64'
sigma_u_dim = len(dolfinx.fem.Function(Vh['sigma_u']).x.array)
sigma_u_dof = np.zeros((num_samples, sigma_u_dim), dtype=dtype)

time_for_solving_PDEs = []
for i in tqdm(range(num_samples)):
    start_time = time.time()
    p = dolfinx.fem.Function(poisson_least_squares.Vh['p'], dtype=dtype)
    p.x.array[:] = p_dof[i,:]
    sigma_u = poisson_least_squares.solve_sigma_u(p=p)
    sigma_u_dof[i,:] = sigma_u.x.array
    end_time = time.time()
    time_for_solving_PDEs.append(end_time - start_time)

100%|██████████| 100/100 [00:37<00:00,  2.64it/s]


In [27]:
print(f'Solved for {num_samples} samples in {np.sum(time_for_solving_PDEs):.2f} seconds')
print(f'average time per sample: {np.mean(time_for_solving_PDEs):.2f} seconds')

Solved for 100 samples in 37.73 seconds
average time per sample: 0.38 seconds


In [14]:
# Solve auxiliary problems for finer space
poisson_least_squares_finer.q = poisson_least_squares_finer.solve_q()
poisson_least_squares_finer.w = poisson_least_squares_finer.solve_w()

In [15]:
dtype = 'float64'
sigma_u_finer_dim = len(dolfinx.fem.Function(finer_Vh['sigma_u']).x.array)
sigma_u_finer_dof = np.zeros((num_samples, sigma_u_finer_dim), dtype=dtype)

time_for_solving_PDEs_finer = []
for i in tqdm(range(num_samples)):
    start_time = time.time()
    p_finer = dolfinx.fem.Function(poisson_least_squares_finer.Vh['p'], dtype=dtype)
    p_finer.x.array[:] = p_dof[i,:]
    sigma_u_finer = poisson_least_squares_finer.solve_sigma_u(p=p_finer)
    sigma_u_finer_dof[i,:] = sigma_u_finer.x.array
    end_time = time.time()
    time_for_solving_PDEs_finer.append(end_time - start_time)

100%|██████████| 100/100 [03:19<00:00,  1.99s/it]


In [16]:
print(f'Solved for {num_samples} samples in {np.sum(time_for_solving_PDEs_finer):.2f} seconds')
print(f'average time per sample: {np.mean(time_for_solving_PDEs_finer):.2f} seconds')

Solved for 100 samples in 199.29 seconds
average time per sample: 1.99 seconds


In [29]:
compute_squared_hdiv_h1_norm = poisson_least_squares.compute_squared_hdiv_h1_norm

In [30]:
torch_dtype = {
    'float32': torch.float32,
    'float64': torch.float64
}

In [33]:
squared_error_list = []
reference_loss_list = []
time_for_assembling_weight = []
for test_index in tqdm(range(num_samples)):
    sigma_u_fc = dolfinx.fem.Function(Vh['sigma_u'])
    sigma_u_fc.x.array[:] = sigma_u_dof[test_index]
    sigma_fc = sigma_u_fc.sub(0).collapse()
    u_fc = sigma_u_fc.sub(1).collapse()

    sigma_u_finer_fc = dolfinx.fem.Function(finer_Vh['sigma_u'])
    sigma_u_finer_fc.x.array[:] = sigma_u_finer_dof[test_index]
    sigma_finer_fc = sigma_u_finer_fc.sub(0).collapse()
    u_finer_fc = sigma_u_finer_fc.sub(1).collapse()

    squared_error = compute_squared_hdiv_h1_norm(sigma_fc - sigma_finer_fc, u_fc - u_finer_fc)

    p_fc = dolfinx.fem.Function(Vh['p'])  
    p_fc.x.array[:] = p_dof[test_index]

    start_time = time.time()
    weight = poisson_least_squares.compute_weight(p_fc)
    end_time = time.time()
    time_for_assembling_weight.append(end_time - start_time)

    weight_tensor = convert_weight_to_tensor(weight, dtype=torch_dtype['float64'])

    y = sigma_u_dof[test_index]
    y = torch.tensor(y, dtype=torch_dtype['float64'])
    reference_loss = torch.dot(y, weight_tensor['A00'] @ y) + 2*torch.dot(y, weight_tensor['A01'])  + weight_tensor['A11']

    squared_error_list.append(squared_error)
    reference_loss_list.append(reference_loss.item())

100%|██████████| 100/100 [00:35<00:00,  2.84it/s]


In [34]:
print(f'# DoFs of sigma_u: {sigma_u_dof.shape[1]}')
print(f'# DoFs of finer sigma_u: {sigma_u_finer_dof.shape[1]}')
print(f'average time for assembling weight: {np.round(np.mean(time_for_assembling_weight), decimals=3)} seconds')
print(f'mean squared error between coarse and finer solutions: {np.mean(squared_error_list):.2e}')
print(f'mean reference loss: {np.mean(reference_loss_list):.2e}')

# DoFs of sigma_u: 57857
# DoFs of finer sigma_u: 214017
average time for assembling weight: 0.051 seconds
mean squared error between coarse and finer solutions: 4.55e-04
mean reference loss: 3.52e-04
